# RSNA EXP-006 OA lexicon + Axial fold0

唯一变量: labels EXP-005 plumbing → **EXP-006 + OA lexicon**（\boa\b / chondrosis / compartm typo）
训练: 与 EXP-004/005 **Axial fold0 完全一致**
对照: EXP-005 LB=**0.699** / EXP-004 Axial=0.687
- rsna_v3 `1612f21b81b0abb56735a5675504043aced36a468c25328004d7bf35f5f02630`
- labels `b39031cf11b3df278957c4fb17e8bea38aa2d7881209c84a742e7c7ac222475c`
- 本地: Medial OA eff 1302→1457, pos 888→996；Lateral OA 1214→1332；PF OA pos 1308→1488
- 短语: `OA of medial compartment` → 0.845；`compartmens` typo → GLOBAL_OA 0.75
- 不自动提交；TIMING 以 meta.json 为准


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: competition mount + Axial coverage
# Correct layout: test_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
import os, csv
from collections import defaultdict

base = '/kaggle/input'

def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b):
        return None
    try:
        entries = sorted(os.listdir(b))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r:
                return r
    return None

root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载')
print('root =', root)

def count_dcm(dirpath):
    if not os.path.isdir(dirpath):
        return -1
    return sum(1 for f in os.listdir(dirpath) if f.lower().endswith('.dcm'))

def plane_stats(series_csv, series_dir, label):
    if not os.path.isfile(series_csv):
        print(label, 'series csv missing', series_csv)
        return
    with open(series_csv, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise SystemExit(f'FAIL: {label} series csv empty')
    plane_key = None
    for k in rows[0].keys():
        if 'plane' in k.lower() or 'anatom' in k.lower():
            plane_key = k
            break
    uid_key = 'StudyInstanceUID'
    ser_key = 'SeriesInstanceUID'
    series_counts = defaultdict(int)
    study_sets = defaultdict(set)
    axial_series_rows = []
    for r in rows:
        pl = r.get(plane_key, '?') if plane_key else '?'
        series_counts[pl] += 1
        if uid_key in r:
            study_sets[pl].add(r[uid_key])
        if pl == 'Axial' and ser_key in r:
            axial_series_rows.append(r)
    print(label, 'plane series counts', dict(series_counts))
    print(label, 'plane study counts', {k: len(v) for k, v in study_sets.items()})
    print(label, 'Axial studies', len(study_sets.get('Axial', set())))
    assert len(study_sets.get('Axial', set())) > 0, f'FAIL: {label} 无 Axial study'

    # Walk both possible layouts
    n_dcm = 0
    hits = 0
    misses = []
    for r in axial_series_rows:
        suid = r[uid_key]
        seid = r[ser_key]
        candidates = [
            os.path.join(series_dir, suid, seid),
            os.path.join(series_dir, seid),
            os.path.join(series_dir, suid, 'series', seid),
        ]
        found = None
        for c in candidates:
            k = count_dcm(c)
            if k and k > 0:
                found = c
                n_dcm += k
                hits += 1
                print(f'  axial OK study={suid[:14]} series={seid[:16]} dcm={k} path={c}')
                break
        if found is None:
            # fallback: list study dir
            sdir = os.path.join(series_dir, suid)
            if os.path.isdir(sdir):
                children = sorted(os.listdir(sdir))[:8]
                print(f'  axial miss study={suid[:14]} series={seid[:16]} study_children={children}')
            else:
                print(f'  axial miss study={suid[:14]} series={seid[:16]} NO_STUDY_DIR {sdir}')
            misses.append(seid)
    print(label, f'axial series hits={hits}/{len(axial_series_rows)} dcm_total={n_dcm}')
    assert hits > 0 and n_dcm > 0, f'FAIL: {label} Axial DICOM 为空 (hits={hits}, dcm={n_dcm})'

    # also sample sagittal for sanity
    sag = [r for r in rows if (r.get(plane_key) if plane_key else '') == 'Sagittal']
    if sag:
        r0 = sag[0]
        for c in [os.path.join(series_dir, r0[uid_key], r0[ser_key]), os.path.join(series_dir, r0[ser_key])]:
            k = count_dcm(c)
            if k and k > 0:
                print(label, 'sagittal sanity', r0[ser_key][:16], 'dcm', k)
                break

plane_stats(os.path.join(root, 'test_series.csv'), os.path.join(root, 'test_series'), 'test')
plane_stats(os.path.join(root, 'train_series.csv'), os.path.join(root, 'train_series'), 'train')
print('preflight axial: OK')


In [ ]:
import base64, os, hashlib
parts=[]
parts.append("...")
path="/kaggle/working/rsna_v3.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("rsna_v3",h); assert h=="1612f21b81b0abb56735a5675504043aced36a468c25328004d7bf35f5f02630"; assert "_fold_dir" in open(path,encoding="utf-8").read()
print("v3 OK")


In [ ]:
import base64, os, hashlib
parts=[]
parts.append("...")
path="/kaggle/working/rsna_report_labels.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("labels006",h); assert h=="b39031cf11b3df278957c4fb17e8bea38aa2d7881209c84a742e7c7ac222475c"
src=open(path,encoding="utf-8").read()
assert "EXP-006" in src or "oa lexicon" in src.lower() or "chondros" in src
assert "__soft" in src
print("labels006 OK")


In [ ]:
import subprocess, sys, os, json
os.chdir("/kaggle/working")
cmd=[sys.executable,"-u","rsna_v3.py","--planes","axial","--epochs","3","--folds","5","--only-fold","0",
     "--batch","8","--n-clips","2","--experiment-id","exp_006_oa_lexicon","--verbose"]
print("=== EXP-006 OA lexicon + Axial fold0 ===", flush=True)
print(" ".join(cmd), flush=True)
r=subprocess.run(cmd)
print("returncode", r.returncode, flush=True)
import numpy as np, pandas as pd
root="/kaggle/working/artifacts/exp_006_oa_lexicon"
canon=None
for p in [os.path.join(root,"test_uid.csv"), os.path.join(root,"fold0","Axial","test_uid.csv")]:
    if os.path.isfile(p):
        canon=pd.read_csv(p)["StudyInstanceUID"].tolist(); print("uids",len(canon),p); break
d=os.path.join(root,"fold0","Axial")
ok=all(os.path.isfile(os.path.join(d,n)) for n in ["test_pred.npy","test_uid.csv","meta.json"])
print("[artifact]", d, "PASS" if ok else "MISSING")
if ok:
    meta=json.load(open(os.path.join(d,"meta.json"),encoding="utf-8"))
    pred=np.load(os.path.join(d,"test_pred.npy"))
    uids=pd.read_csv(os.path.join(d,"test_uid.csv"))["StudyInstanceUID"].tolist()
    print("meta", {k:meta.get(k) for k in ["fold","plane","completion_status","fold_train_time_sec","prediction_shape"]})
    print("pred", pred.shape, "uid_match", uids==canon, "finite", bool(np.isfinite(pred).all()), "mean", float(pred.mean()))
    assert meta.get("completion_status")=="done"
if os.path.isfile("/kaggle/working/submission.csv"):
    s=pd.read_csv("/kaggle/working/submission.csv")
    print("[sub]", s.shape, "NaN", int(s.isna().sum().sum()))
print("[exp006] done")
